# Capitolo 4 — Leakage, classi sbilanciate, ROC e PR: Bank Marketing (§ 4.10–4.11)
45 211 telefonate, 11.7% di sì. La colonna `duration` viene tolta: si conosce solo dopo la telefonata.

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn

def fai_modello(ingressi, nascosti=256, dropout=0.0):
    return nn.Sequential(nn.Linear(ingressi, nascosti), nn.ReLU(), nn.Dropout(dropout),
                         nn.Linear(nascosti, nascosti), nn.ReLU(), nn.Dropout(dropout), nn.Linear(nascosti, 1))

def addestra(modello, Xtr, ytr, Xva, yva, epoche=300, lr=1e-3, weight_decay=0.0, pazienza=None, pos_weight=None):
    perdita_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight) if pos_weight else None)
    opt = torch.optim.Adam(modello.parameters(), lr=lr, weight_decay=weight_decay)
    Xtr, ytr, Xva, yva = map(torch.from_numpy, (Xtr, ytr, Xva, yva))
    storia = {"train": [], "val": []}; migliore = {"perdita": float("inf"), "pesi": None, "epoca": 0}; attesa = 0
    for epoca in range(epoche):
        modello.train(); opt.zero_grad()
        perdita = perdita_fn(modello(Xtr).squeeze(1), ytr); perdita.backward(); opt.step()
        modello.eval()
        with torch.no_grad(): perdita_val = nn.BCEWithLogitsLoss()(modello(Xva).squeeze(1), yva).item()
        storia["train"].append(perdita.item()); storia["val"].append(perdita_val)
        if perdita_val < migliore["perdita"]:
            migliore = {"perdita": perdita_val, "epoca": epoca, "pesi": {k: v.clone() for k, v in modello.state_dict().items()}}; attesa = 0
        else:
            attesa += 1
            if pazienza is not None and attesa >= pazienza: break
    modello.load_state_dict(migliore["pesi"])
    return storia, migliore

def probabilita(modello, X):
    modello.eval()
    with torch.no_grad(): return torch.sigmoid(modello(torch.from_numpy(X)).squeeze(1)).numpy()

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve, average_precision_score)
fissa_seme(42)
bk = dati.bank_marketing()
print(bk.shape, bk["y"].value_counts(normalize=True).round(3).to_dict())
bk = bk.drop(columns=["duration"])                       # leakage

## Preparazione: one-hot delle 9 categoriche, divisione stratificata, normalizzazione

In [ ]:
y = (bk["y"] == "yes").astype(np.float32).values
cat = [c for c in bk.columns if not pd.api.types.is_numeric_dtype(bk[c]) and c != "y"]
Xdf = bk.drop(columns="y")
d_tr, d_tmp, y_tr, y_tmp = train_test_split(Xdf, y, test_size=0.3, random_state=42, stratify=y)
d_va, d_te, y_va, y_te = train_test_split(d_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)
d_tr = pd.get_dummies(d_tr, columns=cat, dtype=float)
d_va = pd.get_dummies(d_va, columns=cat, dtype=float).reindex(columns=d_tr.columns, fill_value=0.0)
d_te = pd.get_dummies(d_te, columns=cat, dtype=float).reindex(columns=d_tr.columns, fill_value=0.0)
scaler = StandardScaler().fit(d_tr.values)
X_tr, X_va, X_te = [scaler.transform(d.values).astype(np.float32) for d in (d_tr, d_va, d_te)]
print("colonne:", X_tr.shape[1], "| campioni", len(y_tr), len(y_va), len(y_te), "| positivi nel test", int(y_te.sum()))

## Tre modelli: standard, perdita pesata, oversampling

In [ ]:
def prova(nome, X=X_tr, y=y_tr, **kw):
    fissa_seme(42); m = fai_modello(X.shape[1], nascosti=64, dropout=0.3)
    addestra(m, X, y, X_va, y_va, weight_decay=1e-3, pazienza=30, epoche=400, **kw)
    p = probabilita(m, X_te); pred = (p > 0.5).astype(int)
    print(f"{nome:14s} acc {accuracy_score(y_te, pred):.1%}  prec {precision_score(y_te, pred):.1%}  recall {recall_score(y_te, pred):.1%}  F1 {f1_score(y_te, pred):.3f}  AUC {roc_auc_score(y_te, p):.3f}  AP {average_precision_score(y_te, p):.3f}")
    return m, p

m_std, p_std = prova("standard")
w_pos = float((y_tr == 0).sum() / (y_tr == 1).sum()); print("pos_weight =", round(w_pos, 2))
m_pes, p_pes = prova("pesato", pos_weight=w_pos)
rng = np.random.default_rng(42); pos = np.where(y_tr == 1)[0]; rep = rng.choice(pos, (y_tr == 0).sum() - len(pos), replace=True)
m_ov, p_ov = prova("oversampling", X=np.concatenate([X_tr, X_tr[rep]]), y=np.concatenate([y_tr, y_tr[rep]]))

## La soglia scelta sul validation set (massimo F1)

In [ ]:
p_va = probabilita(m_std, X_va)
soglia = max((f1_score(y_va, (p_va > t).astype(int)), t) for t in np.arange(0.05, 0.6, 0.01))[1]
pred = (p_std > soglia).astype(int)
print(f"soglia {soglia:.2f}: acc {accuracy_score(y_te, pred):.1%} prec {precision_score(y_te, pred):.1%} recall {recall_score(y_te, pred):.1%} F1 {f1_score(y_te, pred):.3f}")
print(confusion_matrix(y_te, pred))

## Curve ROC e precisione–recall del modello standard

In [ ]:
fpr, tpr, _ = roc_curve(y_te, p_std); prec, rec, _ = precision_recall_curve(y_te, p_std)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(fpr, tpr, lw=2, label=f"AUC {roc_auc_score(y_te, p_std):.3f}"); ax[0].plot([0, 1], [0, 1], "--", color="gray"); ax[0].set_xlabel("falsi positivi"); ax[0].set_ylabel("recall"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(rec, prec, lw=2, label=f"AP {average_precision_score(y_te, p_std):.3f}"); ax[1].axhline(y_te.mean(), ls="--", color="gray"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precisione"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()